# Orgupdate (Apify) — remote jobs notebook

This notebook runs Orgupdate’s Apify actors (job scrapers) and downloads the resulting dataset items.

## What you need

- An Apify account
- An API token (set in `APIFY_TOKEN` below)

## References

- GitHub orgupdate repos list: `https://github.com/orgupdate?tab=repositories`
- LinkedIn actor: `https://github.com/orgupdate/Apify-Linkedin-Jobs-Scraper`
- Apify Python client example: `https://docs.apify.com/api/client/python/docs/examples/retrieve-actor-data`

We’ll call each actor, then fetch items from the run’s default dataset and normalize fields into a common schema.


In [ ]:
# Install deps into the active kernel
%pip -q install apify-client pandas

import os
import re
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional

import pandas as pd
from apify_client import ApifyClient

# Put your token here (or set it as an env var before starting Jupyter)
APIFY_TOKEN = os.environ.get("APIFY_TOKEN", "")

if not APIFY_TOKEN:
    raise RuntimeError(
        "Missing APIFY_TOKEN. Set it as an environment variable (recommended) or paste it into this cell."
    )

client = ApifyClient(APIFY_TOKEN)
print("Apify client ready.")


In [ ]:
# Configure which orgupdate actors to run
#
# You can add more from the orgupdate org, e.g.
# - orgupdate/indeed-jobs-scraper
# - orgupdate/workday-job-scraper
# - orgupdate/we-work-remotely-jobs-scraper (if you use the API actor)

ACTORS = [
    {
        "label": "LinkedIn (orgupdate)",
        "actor_id": "orgupdate/linkedin-jobs-scraper",
        "run_input": {
            "includeKeyword": "data analyst",
            "locationName": "Remote",
            "datePosted": "week",
            "pagesToFetch": 1,
            # Optional knobs:
            # "countryName": "india",
            # "jobType": "FULLTIME",
        },
    },
    # {
    #     "label": "Indeed (orgupdate)",
    #     "actor_id": "orgupdate/indeed-jobs-scraper",
    #     "run_input": {
    #         "includeKeyword": "data analyst",
    #         "locationName": "Remote",
    #         "datePosted": "week",
    #         "pagesToFetch": 1,
    #     },
    # },
    # {
    #     "label": "Workday (orgupdate)",
    #     "actor_id": "orgupdate/workday-job-scraper",
    #     "run_input": {
    #         "includeKeyword": "data analyst",
    #         "locationName": "Remote",
    #         "datePosted": "week",
    #         "pagesToFetch": 1,
    #     },
    # },
]

print("Configured actors:")
for a in ACTORS:
    print("-", a["label"], "→", a["actor_id"])


In [ ]:
def _pick_first(*vals: Any) -> Optional[str]:
    for v in vals:
        if v is None:
            continue
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None


def _to_iso8601(dt_like: Any) -> Optional[str]:
    if dt_like is None:
        return None
    if isinstance(dt_like, str):
        s = dt_like.strip()
        if not s:
            return None
        # Already ISO-ish
        if re.match(r"^\d{4}-\d{2}-\d{2}", s):
            return s
        return s
    return None


def normalize_orgupdate_item(item: Dict[str, Any], actor_label: str, actor_id: str) -> Dict[str, Any]:
    """Best-effort normalization across orgupdate actors.

    Fields vary by actor; we only standardize a small set.
    """
    title = _pick_first(item.get("job_title"), item.get("title"), item.get("position"))
    company = _pick_first(item.get("company_name"), item.get("company"), item.get("companyName"))
    location = _pick_first(item.get("location"), item.get("job_location"), item.get("locationName"))

    url = _pick_first(
        item.get("job_url"),
        item.get("jobUrl"),
        item.get("url"),
        item.get("apply_url"),
        item.get("applyUrl"),
    )

    posted_at = _pick_first(
        item.get("date"),
        item.get("date_posted"),
        item.get("datePosted"),
        item.get("published_at"),
    )

    salary = _pick_first(item.get("salary"), item.get("salary_range"), item.get("salaryRange"))
    job_type = _pick_first(item.get("job_type"), item.get("jobType"))

    return {
        "title": title,
        "company": company,
        "location": location,
        "url": url,
        "posted_at": _to_iso8601(posted_at),
        "salary": salary,
        "job_type": job_type,
        "source": f"apify:{actor_id}",
        "actor_label": actor_label,
        "raw": item,
    }


In [ ]:
def run_actor_and_get_items(actor_id: str, run_input: Dict[str, Any], limit: int = 200) -> List[Dict[str, Any]]:
    """Run an Apify actor and return up to `limit` dataset items."""
    run = client.actor(actor_id).call(run_input=run_input)
    dataset_id = run.get("defaultDatasetId")
    if not dataset_id:
        raise RuntimeError(f"No defaultDatasetId in run response for {actor_id}: {run}")

    items: List[Dict[str, Any]] = []
    offset = 0
    page_size = min(250, max(1, limit))

    while offset < limit:
        res = client.dataset(dataset_id).list_items(limit=min(page_size, limit - offset), offset=offset)
        batch = list(res.items or [])
        if not batch:
            break
        items.extend(batch)
        offset += len(batch)
        if len(batch) < page_size:
            break

    return items


all_rows: List[Dict[str, Any]] = []

for a in ACTORS:
    label = a["label"]
    actor_id = a["actor_id"]
    run_input = a["run_input"]

    print(f"Running: {label} ({actor_id}) …")
    raw_items = run_actor_and_get_items(actor_id, run_input, limit=200)
    print(f"  got {len(raw_items)} items")

    for it in raw_items:
        all_rows.append(normalize_orgupdate_item(it, actor_label=label, actor_id=actor_id))

print(f"Total normalized rows: {len(all_rows)}")

df = pd.DataFrame(all_rows)
# Keep the raw payload but put it at the end
cols = [c for c in df.columns if c != "raw"] + (["raw"] if "raw" in df.columns else [])
df = df[cols]
df.head(10)


In [ ]:
# Export helpers
out_dir = os.path.join(os.getcwd(), "out")
os.makedirs(out_dir, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
json_path = os.path.join(out_dir, f"orgupdate_jobs_{stamp}.json")
csv_path = os.path.join(out_dir, f"orgupdate_jobs_{stamp}.csv")

# JSON (raw included)
df.to_json(json_path, orient="records", force_ascii=False, indent=2)
# CSV (no raw column)
df.drop(columns=["raw"], errors="ignore").to_csv(csv_path, index=False)

print("Wrote:")
print("-", json_path)
print("-", csv_path)
